# Day 09. Exercise 02
# Metrics

## 0. Imports

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import joblib

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [ ]:
csv_path = "../data/day-of-week-not-scaled.csv"
df = pd.read_csv(csv_path)

In [ ]:
df_prev = pd.read_csv("../data/dayofweek.csv")
df['dayofweek'] = df_prev['dayofweek']

In [ ]:
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

svm = SVC(
    C=10,
    class_weight=None,
    gamma='auto',
    kernel='rbf',
    probability=True,
    random_state=21
)

svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)
y_proba = svm.predict_proba(X_test)

print(f"accuracy is {accuracy_score(y_test, y_pred):.5f}")
print(f"precision is {precision_score(y_test, y_pred, average='weighted'):.5f}")
print(f"recall is {recall_score(y_test, y_pred, average='weighted'):.5f}")
print(f"roc_auc is {roc_auc_score(y_test, y_proba, multi_class='ovo', average='weighted'):.5f}")

## 3. Decision tree

1. The same task for decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    class_weight='balanced',
    criterion='gini',
    max_depth=21,
    random_state=21
)

dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)
y_proba = dt.predict_proba(X_test)

print(f"accuracy is {accuracy_score(y_test, y_pred):.5f}")
print(f"precision is {precision_score(y_test, y_pred, average='weighted'):.5f}")
print(f"recall is {recall_score(y_test, y_pred, average='weighted'):.5f}")
print(f"roc_auc is {roc_auc_score(y_test, y_proba, multi_class='ovo', average='weighted'):.5f}")

## 4. Random forest

1. The same task for random forest.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    class_weight='balanced',
    criterion='entropy',
    max_depth=24,
    n_estimators=100,
    random_state=21
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)

print(f"accuracy is {accuracy_score(y_test, y_pred):.5f}")
print(f"precision is {precision_score(y_test, y_pred, average='weighted'):.5f}")
print(f"recall is {recall_score(y_test, y_pred, average='weighted'):.5f}")
print(f"roc_auc is {roc_auc_score(y_test, y_proba, multi_class='ovo', average='weighted'):.5f}")

## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [ ]:
best_model = rf

In [ ]:
y_pred_full = best_model.predict(X)
df_errors = df.copy()
df_errors['predicted'] = y_pred_full
df_errors['is_error'] = df_errors['dayofweek'] != df_errors['predicted']

In [ ]:
weekday_err = df_errors.groupby('dayofweek')['is_error'].mean() * 100
print('Error rate by weekday (%):')
print(weekday_err.round(2).sort_values(ascending=False))

worst_day = weekday_err.idxmax()
print(f"\nWorst predicted weekday: {worst_day} → {weekday_err.max():.2f}% errors")

In [ ]:
lab_cols = [col for col in df_errors.columns if col.startswith('labname_')]

df_errors['labname'] = df_errors[lab_cols].idxmax(axis=1)

df_errors['labname'] = df_errors['labname'].str.replace('labname_', '')

In [ ]:
lab_err = df_errors.groupby('labname')['is_error'].mean() * 100

print('Error rate by labname (top 5 worst):')
print(lab_err.round(2).sort_values(ascending=False).head())

worst_lab = lab_err.idxmax()
print(f"Worst labname: {worst_lab} → {lab_err.max():.2f}% errors")

In [ ]:
uid_cols = [col for col in df_errors.columns if col.startswith('uid_')]

df_errors['uid'] = df_errors[uid_cols].idxmax(axis=1)

df_errors['uid'] = df_errors['uid'].str.replace('uid_', '')

In [ ]:
if 'uid' in df_errors.columns:
    user_err = df_errors.groupby('uid')['is_error'].mean() * 100
    print('Error rate by users (top 5 worst):')
    print(user_err.round(2).sort_values(ascending=False).head())

    worst_user = user_err.idxmax()
    print(f"Worst user: {worst_user} → {user_err.max():.2f}% errors")
else:
    print("\nColumn 'uid' not found → user-level analysis skipped")

In [ ]:
joblib.dump(best_model, 'best_model.joblib')
print('Model saved')

## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [ ]:
def evaluate_models(models_with_params):

    results = {}

    for model_class, params in models_with_params:
        model_name = model_class.__name__
        model = model_class(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        results[model_name] = {
            'accuracy':  round(accuracy_score(y_test, y_pred), 5),
            'precision': round(precision_score(y_test, y_pred, average='weighted'), 5),
            'recall':    round(recall_score(y_test, y_pred, average='weighted'), 5),
            'roc_auc':   round(roc_auc_score(y_test, y_proba, multi_class='ovo', average='weighted'), 5)
        }

    return results

In [ ]:
models_list = [
    (SVC, {
        'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf',
        'probability': True, 'random_state': 21
    }),
    (DecisionTreeClassifier, {
        'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21,
        'random_state': 21
    }),
    (RandomForestClassifier, {
        'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 24,
        'n_estimators': 100, 'random_state': 21
    })
]

print(evaluate_models(models_list))